# Advanced Variable Selection (Consistent Framework)
All methods implemented with comparable evaluation metrics (R², Adj R², AIC, BIC)

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import itertools

from sklearn.linear_model import LassoCV, RidgeCV, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

from statsmodels.stats.outliers_influence import variance_inflation_factor

from src.data_prep import prepare_data
from src.features import run_feature_engineering

## Load Data

In [ ]:
df = prepare_data()
df = run_feature_engineering(df)

## Feature Space

In [ ]:
features = [
    'price','freight_value','delivery_time','lag_price','lag_sales',
    'discount_ratio','month','day_of_week'
]

X_full = df[features].fillna(0)
y = df['profit']

# Helper Function (Unified Evaluation)

In [ ]:
def evaluate_subset(X, y, cols):
    X_sub = sm.add_constant(X[list(cols)])
    model = sm.OLS(y, X_sub).fit()

    return {
        'features': cols,
        'n_features': len(cols),
        'r2': model.rsquared,
        'adj_r2': model.rsquared_adj,
        'AIC': model.aic,
        'BIC': model.bic
    }

# 1. Exhaustive Search (Reference)

In [ ]:
results_exhaustive = []

for k in range(1, len(features)+1):
    for combo in itertools.combinations(features, k):
        results_exhaustive.append(evaluate_subset(X_full, y, combo))

exhaustive_df = pd.DataFrame(results_exhaustive)
exhaustive_df.sort_values('AIC').head(10)

# 2. Forward Selection (AIC-based, tracked)

In [ ]:
selected = []
remaining = list(features)
forward_results = []

while remaining:
    scores = []

    for candidate in remaining:
        cols = selected + [candidate]
        res = evaluate_subset(X_full, y, cols)
        scores.append(res)

    best = sorted(scores, key=lambda x: x['AIC'])[0]

    selected.append([f for f in best['features'] if f not in selected][0])
    remaining = [f for f in remaining if f not in selected]

    forward_results.append(best)

pd.DataFrame(forward_results)

# 3. Backward Elimination (tracked)

In [ ]:
cols = features.copy()
backward_results = []

while len(cols) > 1:
    res = evaluate_subset(X_full, y, cols)
    backward_results.append(res)

    X_temp = sm.add_constant(X_full[cols])
    model = sm.OLS(y, X_temp).fit()

    pvals = model.pvalues.drop('const')
    worst = pvals.idxmax()

    cols.remove(worst)

pd.DataFrame(backward_results)

# 4. VIF (with ranking)

In [ ]:
vif = pd.DataFrame()
vif['feature'] = X_full.columns
vif['VIF'] = [variance_inflation_factor(X_full.values, i) for i in range(X_full.shape[1])]
vif.sort_values('VIF', ascending=False)

# 5. Lasso (evaluate selected subset)

In [ ]:
lasso = LassoCV(cv=5)
lasso.fit(X_full, y)

selected_lasso = [f for f, c in zip(features, lasso.coef_) if c != 0]
evaluate_subset(X_full, y, selected_lasso)

# 6. Ridge (full + interpretation)

In [ ]:
ridge = RidgeCV(cv=5)
ridge.fit(X_full, y)
pd.Series(ridge.coef_, index=features)

# 7. Elastic Net

In [ ]:
enet = ElasticNetCV(cv=5)
enet.fit(X_full, y)
selected_enet = [f for f, c in zip(features, enet.coef_) if c != 0]
evaluate_subset(X_full, y, selected_enet)

# 8. Random Forest (top-k subsets evaluation)

In [ ]:
rf = RandomForestRegressor()
rf.fit(X_full, y)

importance_rf = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)

# test top subsets
rf_results = []
for k in range(1, len(features)+1):
    cols = list(importance_rf.head(k).index)
    rf_results.append(evaluate_subset(X_full, y, cols))

pd.DataFrame(rf_results)

# 9. XGBoost (same logic)

In [ ]:
xgb = XGBRegressor()
xgb.fit(X_full, y)

importance_xgb = pd.Series(xgb.feature_importances_, index=features).sort_values(ascending=False)

xgb_results = []
for k in range(1, len(features)+1):
    cols = list(importance_xgb.head(k).index)
    xgb_results.append(evaluate_subset(X_full, y, cols))

pd.DataFrame(xgb_results)

# 10. CatBoost (same logic)

In [ ]:
cat = CatBoostRegressor(verbose=0)
cat.fit(X_full, y)

importance_cat = pd.Series(cat.get_feature_importance(), index=features).sort_values(ascending=False)

cat_results = []
for k in range(1, len(features)+1):
    cols = list(importance_cat.head(k).index)
    cat_results.append(evaluate_subset(X_full, y, cols))

pd.DataFrame(cat_results)

# Final Step
Compare best subsets across ALL methods and select stable variables.